In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from geopy.distance import great_circle
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GCNConv
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Load and Filter Data

In [4]:
users_df = pd.DataFrame()

for chunk in pd.read_json(r"data/yelp/yelp_academic_dataset_user.json", chunksize=100000, lines=True):
    users_df = pd.concat([users_df, chunk], ignore_index = True)

print(users_df.shape)
users_df.head() # 19.8 lakhs

(1987897, 22)


,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,...,13,10,17,3,66,96,119,119,35,18
3,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011","enx1vVPnfdNUdPho6PH_wg, 4wOcvMLtU6a9Lslggq74Vg...",28,...,4,1,6,2,12,16,26,26,10,9
4,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,,"PBK4q9KEEBHhFvSXCUirIw, 3FWPpM7KU1gXeOM_ZbYMbA...",1,...,1,0,0,0,1,1,0,0,0,0


In [5]:
users_df.columns

Index(['user_id', 'name', 'review_count', 'yelping_since', 'useful', 'funny',
       'cool', 'elite', 'friends', 'fans', 'average_stars', 'compliment_hot',
       'compliment_more', 'compliment_profile', 'compliment_cute',
       'compliment_list', 'compliment_note', 'compliment_plain',
       'compliment_cool', 'compliment_funny', 'compliment_writer',
       'compliment_photos'],
      dtype='object')

In [6]:
business_df = pd.DataFrame()

for chunk in pd.read_json(r"data/yelp/yelp_academic_dataset_business.json",chunksize=100000, lines=True):
    business_df = pd.concat([business_df, chunk], ignore_index = True)

print(business_df.shape)
business_df.head()  # 1.5 lakhs

(150346, 14)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


In [7]:
business_df.columns

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours'],
      dtype='object')

In [8]:
reviews_df = pd.DataFrame()

for chunk in pd.read_json(r"data/yelp/yelp_academic_dataset_review.json",chunksize=100000, lines=True):
    reviews_df = pd.concat([reviews_df, chunk], ignore_index = True)

print(reviews_df.shape)
reviews_df.head()

(6990280, 9)


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15


In [9]:
reviews_df.columns

Index(['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny',
       'cool', 'text', 'date'],
      dtype='object')

In [10]:
restaurant_keywords = ['restaurant','food','cafe','diner','bakery','pizza','bar','bistro','bbq','fast food','coffee','tea']
business_df['categories'] = business_df['categories'].astype(str).str.lower()
restaurant_df = business_df[business_df['categories'].apply(lambda x: any(k in x for k in restaurant_keywords))]
restaurant_reviews_df = reviews_df[reviews_df['business_id'].isin(restaurant_df['business_id'])]
filtered_users_df = users_df[['user_id','review_count','average_stars','fans']]

# Content-Based Filtering

In [88]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
import pandas as pd

# Combine features
restaurant_df['combined_features'] = restaurant_df['categories'] + ' ' + restaurant_df['attributes'].astype(str)

# Train-test split
train_df_content, test_df_content = train_test_split(restaurant_df, test_size=0.1, random_state=42)

# Fit TF-IDF and SVD only on training data
tfidf = TfidfVectorizer(stop_words='english', max_features=3000)
tfidf_matrix_train = tfidf.fit_transform(train_df_content['combined_features'])

svd_sk = TruncatedSVD(n_components=100, random_state=42)
tfidf_reduced_train = svd_sk.fit_transform(tfidf_matrix_train)

# Fit Nearest Neighbors model
nn_model = NearestNeighbors(n_neighbors=10, metric='cosine')
nn_model.fit(tfidf_reduced_train)

# Recommendation function that works for both in-train and new descriptions
def content_recommend(name=None, description=None, top_n=5):
    """
    If name is in train_df, use precomputed features.
    If not, use TFIDF-SVD vector from the input description.
    """
    if name is not None and name in train_df_content['name'].values:
        idx = train_df_content[train_df_content['name'] == name].index[0]
        query_vec = tfidf_reduced_train[train_df_content.index.get_loc(idx)].reshape(1, -1)
    elif description is not None:
        tfidf_vec = tfidf.transform([description])
        query_vec = svd_sk.transform(tfidf_vec)
    else:
        raise ValueError("Either a valid restaurant name from training data or a description must be provided.")

    _, inds = nn_model.kneighbors(query_vec, n_neighbors=top_n)
    return train_df_content.iloc[inds[0]][['name', 'stars', 'categories']].reset_index(drop=True)


/tmp/ipykernel_2607053/1723768922.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  restaurant_df['combined_features'] = restaurant_df['categories'] + ' ' + restaurant_df['attributes'].astype(str)


In [89]:
names = test_df_content['name'].tolist()
descriptions = test_df_content['combined_features'].tolist()

print("Few Business Names and Their Descriptions in Test set:\n")
for name, desc in zip(names[:5], descriptions[:5]):
    print(f"{name}:\n  {desc[:50]}...\n")

print(f"Recommending for business name: {names[1]} with description: {descriptions[1]}")
content_recommend('The Gumbo Krewe',description=descriptions[1])

Few Business Names and Their Descriptions in Test set:

Walgreens:
  cosmetics & beauty supply, shopping, convenience s...

The Gumbo Krewe:
  southern, salad, specialty food, cajun/creole, pas...

The Alibi:
  dive bars, restaurants, bars, nightlife {'Restaura...

Voodoo Gumbo:
  cajun/creole, restaurants {'HasTV': 'False', 'Busi...

Jimmy John's:
  food, sandwiches, delis, food delivery services, r...

Recommending for business name: The Gumbo Krewe with description: southern, salad, specialty food, cajun/creole, pasta shops, food, seafood, restaurants, sandwiches {'BikeParking': 'True', 'OutdoorSeating': 'False', 'RestaurantsTakeOut': 'True', 'BusinessAcceptsCreditCards': 'True', 'RestaurantsDelivery': 'None', 'HasTV': 'True', 'RestaurantsReservations': 'True', 'WheelchairAccessible': 'True', 'BYOB': 'False', 'RestaurantsTableService': 'True', 'Alcohol': "u'full_bar'", 'BusinessParking': "{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}", 'Caters':

,name,stars,categories
0,Crafty Crab St. Pete,3.5,"comfort food, cajun/creole, restaurants, seafood"
1,Deep Sea Crab,4.5,"seafood, restaurants, cajun/creole"
2,Streetcar Poboys & Seafood,4.5,"butcher, food, restaurants, cajun/creole, sand..."
3,Cluster Busters,4.0,"cajun/creole, seafood, restaurants"
4,OHot Cajun Seafood,4.5,"restaurants, cajun/creole, seafood"


# Collaborative Filtering (SVD)

In [83]:
from surprise import SVD, Dataset, Reader, accuracy
from sklearn.model_selection import train_test_split
import pandas as pd



train_df_collab, test_df_collab = train_test_split(restaurant_reviews_df, test_size=0.1, random_state=42)

reader = Reader(rating_scale=(1, 5))
train_data_collab = Dataset.load_from_df(train_df_collab[['user_id', 'business_id', 'stars']], reader)
trainset_collab = train_data_collab.build_full_trainset()

svd = SVD()
svd.fit(trainset_collab)

train_user_ids_collab = set(train_df_collab['user_id'])
test_df_collab = test_df_collab[test_df_collab['user_id'].isin(train_user_ids_collab)]

testset_collab = list(test_df_collab[['user_id', 'business_id', 'stars']].itertuples(index=False, name=None))

#predictions_collab = svd.test(testset_collab)
#rmse_collab = accuracy.rmse(predictions_collab)

def cf_recommend(user_id, top_n=5):
    
    if user_id not in train_df_collab['user_id'].values:
        print(f"User {user_id} not in training set. Cannot recommend.")
        return pd.DataFrame()

    user_rated = set(train_df_collab[train_df_collab['user_id'] == user_id]['business_id'])

    preds = [
        (bid, svd.predict(user_id, bid).est)
        for bid in restaurant_df['business_id']
        if bid not in user_rated
    ]
    preds.sort(key=lambda x: x[1], reverse=True)
    top_bids = [bid for bid, _ in preds[:top_n]]

    return restaurant_df[restaurant_df['business_id'].isin(top_bids)][['name', 'stars', 'categories']]




In [84]:
user_id_test=test_df_collab['user_id'].tolist()
print(f"Some user ids in test set whose ratings were not seen during training \n {user_id_test[:6]}")

print(f"Recommendations for user {user_id_test[0]}:")
cf_recommend(user_id_test[0])



Some user ids in test set whose ratings were not seen during training 
 ['Hp3F4zLIpCc2QZ5QxB2U2Q', 'mJy-5ShuwwYRayxRtl6xxA', 'fr1Hz2acAb3OaL3l6DyKNg', 'R2e2nVQKA7bjxrW1nWHt3A', 'KvjM4uBJJmxq25uYVPSCFw', '7qBgGzyf0FbHKC0jiI1J8A']
Recommendations for user Hp3F4zLIpCc2QZ5QxB2U2Q:


,name,stars,categories
10760,Hair Cut & Color Design by Carleen Sanchez,5.0,"shopping, cosmetics & beauty supply, hair exte..."
14550,Novella Wine Bar,5.0,"nightlife, cafes, bars, food, wine bars, resta..."
20252,Cajé Coffee Roasters - Haley St,4.5,"coffee roasteries, acai bowls, food, coffee & ..."
26927,Little Fox,4.5,"wine bars, american (new), restaurants, bars, ..."
29267,Armature Works,4.5,"specialty food, event planning & services, foo..."


# Location-Based Recommendation

In [21]:
def location_recommend(user_lat, user_lon, top_n=5):
    restaurant_df['distance'] = restaurant_df.apply(
        lambda r: great_circle((user_lat, user_lon), (r.latitude, r.longitude)).km, axis=1)
    
    filtered_df = restaurant_df[restaurant_df['distance'] > 0]
    return filtered_df.sort_values('distance').head(top_n)[['name', 'stars', 'categories', 'distance']]


In [22]:
print('Location of some business Ids')
restaurant_df[['business_id','latitude','longitude','categories']].head(5)

Location of some business Ids


,business_id,latitude,longitude,categories
3,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,"restaurants, food, bubble tea, coffee & tea, b..."
4,mWMc6_wTdE0EUBKIGXDVfA,40.338183,-75.471659,"brewpubs, breweries, food"
5,CF33F8-E6oudUQ46HnavjQ,36.269593,-87.058943,"burgers, fast food, sandwiches, food, ice crea..."
8,k0hlBqXX-Bt0vf1op7Jr1w,38.565165,-90.321087,"pubs, restaurants, italian, bars, american (tr..."
9,bBDDEgkFA1Otx9Lfe7BZUQ,36.208102,-86.768170,"ice cream & frozen yogurt, fast food, burgers,..."


In [23]:
lat_lon_pairs = list(zip(restaurant_df['latitude'], restaurant_df['longitude']))

sample_row = restaurant_df.iloc[3]

print(f"\n Location based recommendation for business id : {sample_row['business_id']} with name {sample_row['name']}")
print(f"Whose Location:, lat={sample_row['latitude']}, lon={sample_row['longitude']}")

results = location_recommend(sample_row['latitude'], sample_row['longitude'], top_n=5)
results


 Location based recommendation for business id : k0hlBqXX-Bt0vf1op7Jr1w with name Tsevi's Pub And Grill
Whose Location:, lat=38.5651648, lon=-90.3210868


/tmp/ipykernel_2607053/3261615286.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  restaurant_df['distance'] = restaurant_df.apply(


,name,stars,categories,distance
67662,China Garden,2.5,"chinese, restaurants",0.047687
106725,Sedara Sweets & Ice Cream,4.5,"coffee & tea, ice cream & frozen yogurt, food,...",0.055839
56543,Little Caesars Pizza,3.0,"restaurants, pizza",0.062252
29157,Las Fuentes,3.5,"bars, mexican, restaurants, nightlife",0.075405
134970,Fortel's Pizza Den,3.5,"italian, pizza, restaurants",0.130919


# Popularity-Based Recommendation

In [69]:
def popularity_recommend(df, top_n=5, city=None, cuisine=None, price_range=None, min_rating=None):
    df2 = df.copy()
    if city: df2 = df2[df2.city.str.lower() == city.lower()]
    if cuisine: df2 = df2[df2.categories.str.contains(cuisine, case=False, na=False)]
    if price_range and 'price_range' in df2: df2 = df2[df2.price_range == price_range]
    if min_rating: df2 = df2[df2.stars >= min_rating]
    C = df2.stars.mean()
    m = df2.review_count.quantile(0.75)
    qualified = df2[df2.review_count >= m].copy()
    qualified['popularity_score'] = qualified.apply(
        lambda x: (x.review_count / (x.review_count + m) * x.stars) + (m / (x.review_count + m) * C), axis=1)
    return qualified.sort_values('popularity_score', ascending=False).head(top_n)[
        ['name','city','stars','review_count','categories','popularity_score']]

In [97]:
print('Some city names')
city_names=restaurant_df['city'].unique().tolist()
print(city_names[:10])
print(f'Popularity based recommendation of business based in {city_names[0]} with 3 or more star rating')
popularity_recommend(restaurant_df,city=city_names[0],min_rating=3)

Some city names
['Philadelphia', 'Green Lane', 'Ashland City', 'Affton', 'Nashville', 'Tampa Bay', 'Indianapolis', 'Largo', 'Edmonton', 'Reno']
Popularity based recommendation of business based in Philadelphia with 3 or more star rating


,name,city,stars,review_count,categories,popularity_score
7094,The Sweet Life Bakeshop,Philadelphia,5.0,316,"food, cupcakes, desserts, bakeries",4.709405
67317,ICI Macarons & Cafe,Philadelphia,5.0,276,"desserts, bakeries, food, coffee & tea, specia...",4.679213
69859,Tortilleria San Roman,Philadelphia,5.0,219,"convenience stores, italian, specialty food, m...",4.623466
6287,Free Tours By Foot,Philadelphia,5.0,184,"hotels & travel, walking tours, tours, food tours",4.578488
37473,Philadelphia Distilling,Philadelphia,5.0,182,"distilleries, nightlife, food, beer, wine & sp...",4.575591


# Graph-Based Recommendation (GNN)

In [26]:
import sklearn.model_selection as skms

# user & restaurant ID mappings
user_ids = {uid: i for i, uid in enumerate(restaurant_reviews_df['user_id'].unique())}
restaurant_ids = {bid: i + len(user_ids) for i, bid in enumerate(restaurant_df['business_id'].unique())}

# Build edge list from CF's train/test split
train_edges_list = [(user_ids[row.user_id], restaurant_ids[row.business_id]) for _, row in train_df_collab.iterrows()]
val_edges_list = [(user_ids[row.user_id], restaurant_ids[row.business_id]) for _, row in test_df_collab.iterrows()]

# Convert to PyTorch tensors
train_edges = torch.tensor(train_edges_list, dtype=torch.long).t().contiguous()
val_edges = torch.tensor(val_edges_list, dtype=torch.long).t().contiguous()


# Define GNN model
class GNN(nn.Module):
    def __init__(self, num_nodes, dim=64):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, dim)
        self.conv1 = GCNConv(dim, 128)
        self.conv2 = GCNConv(128, 64)

    def forward(self, edge_index):
        x = self.emb.weight
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model = GNN(len(user_ids) + len(restaurant_ids))
opt = optim.Adam(model.parameters(), lr=0.001)

# Training loop with validation loss
for epoch in range(10):
    model.train()
    opt.zero_grad()
    em = model(train_edges)
    loss = ((em[train_edges[0]] - em[train_edges[1]]) ** 2).mean()
    loss.backward()
    opt.step()

    # Validation loss
    model.eval()
    with torch.no_grad():
        em_val = model(val_edges)
        val_loss = ((em_val[val_edges[0]] - em_val[val_edges[1]]) ** 2).mean()

    print(f"Epoch {epoch+1}, Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

torch.save(model.state_dict(), "gnn_model.pth")

Epoch 1, Train Loss: 45.3271, Val Loss: 3.0254
Epoch 2, Train Loss: 35.7697, Val Loss: 2.4444
Epoch 3, Train Loss: 27.9266, Val Loss: 1.9760
Epoch 4, Train Loss: 21.6056, Val Loss: 1.6046
Epoch 5, Train Loss: 16.5960, Val Loss: 1.3150
Epoch 6, Train Loss: 12.6948, Val Loss: 1.0926
Epoch 7, Train Loss: 9.7028, Val Loss: 0.9238
Epoch 8, Train Loss: 7.4369, Val Loss: 0.7972
Epoch 9, Train Loss: 5.7414, Val Loss: 0.7035
Epoch 10, Train Loss: 4.4920, Val Loss: 0.6357


In [27]:
def graph_recommend(user_id, top_n=5):
    if user_id not in user_ids:
        return pd.DataFrame()
    model.load_state_dict(torch.load("gnn_model.pth"))
    model.eval()
    em = model(train_edges).detach().numpy()
    uvec = em[user_ids[user_id]]
    rvecs = em[len(user_ids):]
    scores = rvecs.dot(uvec)
    top = scores.argsort()[-top_n:][::-1]
    bid_keys = list(restaurant_ids.keys())
    top_ids = [bid_keys[i] for i in top]
    return restaurant_df[restaurant_df.business_id.isin(top_ids)][['name','stars','categories']]

In [28]:
user_id_test=test_df_collab['user_id'].tolist()
print(f"User IDs from the test set whose edges were not used during training:\n{user_id_test[:6]}")
print(f"Recommendations for user {user_id_test[0]}:")
graph_recommend(user_id_test[0])


User IDs from the test set whose edges were not used during training:
['Hp3F4zLIpCc2QZ5QxB2U2Q', 'mJy-5ShuwwYRayxRtl6xxA', 'fr1Hz2acAb3OaL3l6DyKNg', 'R2e2nVQKA7bjxrW1nWHt3A', 'KvjM4uBJJmxq25uYVPSCFw', '7qBgGzyf0FbHKC0jiI1J8A']
Recommendations for user Hp3F4zLIpCc2QZ5QxB2U2Q:


,name,stars,categories
31033,Royal House,4.0,"american (new), restaurants, sandwiches, seafo..."
91757,Hattie B’s Hot Chicken - Nashville,4.5,"american (traditional), chicken shop, southern..."
112552,Oceana Grill,4.0,"restaurants, seafood, cajun/creole, breakfast ..."
113731,Acme Oyster House,4.0,"live/raw food, seafood, restaurants, cajun/creole"
143157,Reading Terminal Market,4.5,"candy stores, shopping, department stores, fas..."


# Final Hybrid Recommendation

name: Name of a restaurant to get content-based recommendations for similar businesses.

description: A description of business to get content-based recommendations, use if name is not available

user_id: ID of the user for collaborative and graph-based personalized recommendations.

lat, lon: User or target location to get nearby restaurants using location-based filtering

city: Name of ciy whose to get popular business based on review count

In [98]:
def hybrid_recommend(user_id=None, name=None, description=None, lat=None, lon=None, city=None, top_n=10):
    common_cols = ['name', 'stars', 'categories']

    # Content-based
    if name or description:
        df1 = content_recommend(name=name, description=description)[common_cols]
    else:
        print("Content-based recommendation skipped: no name or description provided.")
        df1 = pd.DataFrame(columns=common_cols)

    # CF + GNN-based
    if user_id and user_id in user_id_test:
        df2 = cf_recommend(user_id, top_n)[common_cols]
        df3 = graph_recommend(user_id, top_n)[common_cols]
    else:
        print("Collaborative filtering and GNN recommendation skipped: invalid or missing user ID.")
        df2 = pd.DataFrame(columns=common_cols)
        df3 = pd.DataFrame(columns=common_cols)

    # Location-based
    if lat is not None and lon is not None:
        df4 = location_recommend(lat, lon, top_n)[common_cols]
    else:
        print("Location-based recommendation skipped: latitude or longitude not provided.")
        df4 = pd.DataFrame(columns=common_cols)

    # Popularity full set (just once)
    df5_full = popularity_recommend(restaurant_df, city=city)[common_cols]

    # Combine df1 to df4
    combined_part = pd.concat([df1, df2, df3, df4], ignore_index=True).drop_duplicates('name')
    existing_names = combined_part['name'].unique()
    existing_count = len(existing_names)

    # Decide how many from popularity to include
    if existing_count >= top_n:
        df5 = df5_full.head(0)
    else:
        print('Using popularity based recommnedation')
        needed = top_n - existing_count
        df5 = df5_full.head(needed)



    combined = pd.concat([combined_part, df5], ignore_index=True).drop_duplicates('name')
    combined = combined.dropna(subset=['stars'])
    combined['score_norm'] = combined['stars'] / combined['stars'].max()

    return (
        combined.sort_values('score_norm', ascending=False).head(top_n),
        df1, df2, df3, df4, df5
    )



# Hybrid Recommendation based on user id

In [102]:
print(f"User IDs from the test set \n{user_id_test[:6]}")
print(f'Recommending for user: {user_id_test[0]}')
combined, df_content, df_cf, df_gnn, df_loc, df_pop =hybrid_recommend(user_id=user_id_test[0])

print('Showing only Collaborative based recommendation')
df_cf


User IDs from the test set 
['Hp3F4zLIpCc2QZ5QxB2U2Q', 'mJy-5ShuwwYRayxRtl6xxA', 'fr1Hz2acAb3OaL3l6DyKNg', 'R2e2nVQKA7bjxrW1nWHt3A', 'KvjM4uBJJmxq25uYVPSCFw', '7qBgGzyf0FbHKC0jiI1J8A']
Recommending for user: Hp3F4zLIpCc2QZ5QxB2U2Q
Content-based recommendation skipped: no name or description provided.
Location-based recommendation skipped: latitude or longitude not provided.
Showing only Collaborative based recommendation


/tmp/ipykernel_2607053/325654842.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_part = pd.concat([df1, df2, df3, df4], ignore_index=True).drop_duplicates('name')


,name,stars,categories
10760,Hair Cut & Color Design by Carleen Sanchez,5.0,"shopping, cosmetics & beauty supply, hair exte..."
14550,Novella Wine Bar,5.0,"nightlife, cafes, bars, food, wine bars, resta..."
20252,Cajé Coffee Roasters - Haley St,4.5,"coffee roasteries, acai bowls, food, coffee & ..."
26927,Little Fox,4.5,"wine bars, american (new), restaurants, bars, ..."
29267,Armature Works,4.5,"specialty food, event planning & services, foo..."
30677,Delicious Expressions,5.0,"caterers, restaurants, desserts, bakeries, eve..."
46027,20 Shekels Bread,5.0,"bakeries, coffee & tea, breakfast & brunch, or..."
58300,Yolk White & Associates,5.0,"food stands, restaurants, food, coffee & tea, ..."
62898,HopScotch Cafe,5.0,"nightlife, bars, beer bar, restaurants, sandwi..."
78687,Gemelli - Artisanal Gelato & Dessert Cafe,4.5,"bakeries, coffee & tea, gelato, food, desserts"


In [103]:
print(f'Recommending for user: {user_id_test[0]}')
print('Showing only GNN based recommendation')
df_gnn

Recommending for user: Hp3F4zLIpCc2QZ5QxB2U2Q
Showing only GNN based recommendation


,name,stars,categories
31033,Royal House,4.0,"american (new), restaurants, sandwiches, seafo..."
41740,Brennan's,4.0,"cajun/creole, restaurants"
57332,Geno's Steaks,2.5,"sandwiches, cheesesteaks, steakhouses, restaur..."
91757,Hattie B’s Hot Chicken - Nashville,4.5,"american (traditional), chicken shop, southern..."
97331,Cochon,4.0,"cajun/creole, seafood, restaurants"
100024,Mother's Restaurant,3.5,"cajun/creole, restaurants, event planning & se..."
112552,Oceana Grill,4.0,"restaurants, seafood, cajun/creole, breakfast ..."
113731,Acme Oyster House,4.0,"live/raw food, seafood, restaurants, cajun/creole"
143157,Reading Terminal Market,4.5,"candy stores, shopping, department stores, fas..."
147081,Ruby Slipper - New Orleans,4.5,"restaurants, american (traditional), american ..."


In [104]:
print(f'Recommending for user: {user_id_test[0]}')
print('Showing Hybrid recommendation')
combined

Recommending for user: Hp3F4zLIpCc2QZ5QxB2U2Q
Showing Hybrid recommendation


,name,stars,categories,score_norm
0,Hair Cut & Color Design by Carleen Sanchez,5.0,"shopping, cosmetics & beauty supply, hair exte...",1.0
1,Novella Wine Bar,5.0,"nightlife, cafes, bars, food, wine bars, resta...",1.0
5,Delicious Expressions,5.0,"caterers, restaurants, desserts, bakeries, eve...",1.0
6,20 Shekels Bread,5.0,"bakeries, coffee & tea, breakfast & brunch, or...",1.0
7,Yolk White & Associates,5.0,"food stands, restaurants, food, coffee & tea, ...",1.0
8,HopScotch Cafe,5.0,"nightlife, bars, beer bar, restaurants, sandwi...",1.0
9,Gemelli - Artisanal Gelato & Dessert Cafe,4.5,"bakeries, coffee & tea, gelato, food, desserts",0.9
18,Reading Terminal Market,4.5,"candy stores, shopping, department stores, fas...",0.9
13,Hattie B’s Hot Chicken - Nashville,4.5,"american (traditional), chicken shop, southern...",0.9
19,Ruby Slipper - New Orleans,4.5,"restaurants, american (traditional), american ...",0.9


# Hybrid based on restaurant name or description of food/restaurant

In [105]:
names = test_df_content['name'].tolist()
descriptions = test_df_content['combined_features'].tolist()

print("Few Business Names and Their Descriptions in test set:\n")
for name, desc in zip(names[:5], descriptions[:5]):
    print(f"{name}:\n  {desc[:50]}...\n")



Few Business Names and Their Descriptions in test set:

Walgreens:
  cosmetics & beauty supply, shopping, convenience s...

The Gumbo Krewe:
  southern, salad, specialty food, cajun/creole, pas...

The Alibi:
  dive bars, restaurants, bars, nightlife {'Restaura...

Voodoo Gumbo:
  cajun/creole, restaurants {'HasTV': 'False', 'Busi...

Jimmy John's:
  food, sandwiches, delis, food delivery services, r...



In [106]:
print(f"Recommending based on business name: {names[1]} with description: {descriptions[1]}")
combined, df_content, df_cf, df_gnn, df_loc, df_pop =hybrid_recommend(name=names[1],description=descriptions[1])

Recommending based on business name: The Gumbo Krewe with description: southern, salad, specialty food, cajun/creole, pasta shops, food, seafood, restaurants, sandwiches {'BikeParking': 'True', 'OutdoorSeating': 'False', 'RestaurantsTakeOut': 'True', 'BusinessAcceptsCreditCards': 'True', 'RestaurantsDelivery': 'None', 'HasTV': 'True', 'RestaurantsReservations': 'True', 'WheelchairAccessible': 'True', 'BYOB': 'False', 'RestaurantsTableService': 'True', 'Alcohol': "u'full_bar'", 'BusinessParking': "{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}", 'Caters': 'True', 'WiFi': "u'free'", 'HappyHour': 'True'}
Collaborative filtering and GNN recommendation skipped: invalid or missing user ID.
Location-based recommendation skipped: latitude or longitude not provided.
Using popularity based recommnedation


/tmp/ipykernel_2607053/325654842.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_part = pd.concat([df1, df2, df3, df4], ignore_index=True).drop_duplicates('name')


In [107]:
print(f"Recommending based on business name: {names[1]} with description: {descriptions[1]}")
print('Showing only content based')
df_content

Recommending based on business name: The Gumbo Krewe with description: southern, salad, specialty food, cajun/creole, pasta shops, food, seafood, restaurants, sandwiches {'BikeParking': 'True', 'OutdoorSeating': 'False', 'RestaurantsTakeOut': 'True', 'BusinessAcceptsCreditCards': 'True', 'RestaurantsDelivery': 'None', 'HasTV': 'True', 'RestaurantsReservations': 'True', 'WheelchairAccessible': 'True', 'BYOB': 'False', 'RestaurantsTableService': 'True', 'Alcohol': "u'full_bar'", 'BusinessParking': "{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}", 'Caters': 'True', 'WiFi': "u'free'", 'HappyHour': 'True'}
Showing only content based


,name,stars,categories
0,Crafty Crab St. Pete,3.5,"comfort food, cajun/creole, restaurants, seafood"
1,Deep Sea Crab,4.5,"seafood, restaurants, cajun/creole"
2,Streetcar Poboys & Seafood,4.5,"butcher, food, restaurants, cajun/creole, sand..."
3,Cluster Busters,4.0,"cajun/creole, seafood, restaurants"
4,OHot Cajun Seafood,4.5,"restaurants, cajun/creole, seafood"


In [108]:
print(f"Recommending based on business name: {names[1]} with description: {descriptions[1]}")
print('Hybrid recommendation')
combined

Recommending based on business name: The Gumbo Krewe with description: southern, salad, specialty food, cajun/creole, pasta shops, food, seafood, restaurants, sandwiches {'BikeParking': 'True', 'OutdoorSeating': 'False', 'RestaurantsTakeOut': 'True', 'BusinessAcceptsCreditCards': 'True', 'RestaurantsDelivery': 'None', 'HasTV': 'True', 'RestaurantsReservations': 'True', 'WheelchairAccessible': 'True', 'BYOB': 'False', 'RestaurantsTableService': 'True', 'Alcohol': "u'full_bar'", 'BusinessParking': "{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}", 'Caters': 'True', 'WiFi': "u'free'", 'HappyHour': 'True'}
Hybrid recommendation


,name,stars,categories,score_norm
5,Blues City Deli,5.0,"delis, bars, restaurants, nightlife, pubs, ame...",1.0
6,Carlillos Cocina,5.0,"bars, mexican, breakfast & brunch, restaurants...",1.0
7,Tumerico,5.0,"mexican, gluten-free, vegetarian, restaurants,...",1.0
8,Yats,5.0,"cajun/creole, restaurants, caterers, comfort f...",1.0
9,Nelson's Green Brier Distillery,5.0,"distilleries, tours, hotels & travel, historic...",1.0
1,Deep Sea Crab,4.5,"seafood, restaurants, cajun/creole",0.9
2,Streetcar Poboys & Seafood,4.5,"butcher, food, restaurants, cajun/creole, sand...",0.9
4,OHot Cajun Seafood,4.5,"restaurants, cajun/creole, seafood",0.9
3,Cluster Busters,4.0,"cajun/creole, seafood, restaurants",0.8
0,Crafty Crab St. Pete,3.5,"comfort food, cajun/creole, restaurants, seafood",0.7


# Location Based Hybrid recommendation

In [109]:
combined, df_content, df_cf, df_gnn, df_loc, df_pop =hybrid_recommend(lat=lat_lon_pairs[0][0],lon=lat_lon_pairs[0][1])
print(f"Location:, lat={sample_row['latitude']}, lon={sample_row['longitude']}")
combined

Content-based recommendation skipped: no name or description provided.
Collaborative filtering and GNN recommendation skipped: invalid or missing user ID.


/tmp/ipykernel_2607053/3261615286.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  restaurant_df['distance'] = restaurant_df.apply(


Location:, lat=38.5651648, lon=-90.3210868


/tmp/ipykernel_2607053/325654842.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_part = pd.concat([df1, df2, df3, df4], ignore_index=True).drop_duplicates('name')


,name,stars,categories,score_norm
4,Cily Chicken Rice,4.5,"hainan, fast food, chinese, coffee & tea, food...",1.000000
1,Good Harvest 大丰收,4.0,"cantonese, seafood, chinese, restaurants",0.888889
2,Ocean Harmony,4.0,"asian fusion, seafood, chinese, restaurants",0.888889
5,About BBQ,4.0,"barbeque, restaurants",0.888889
6,Canto House,4.0,"restaurants, chinese, cantonese, barbeque, noo...",0.888889
7,T.C. Palate,4.0,"chinese, restaurants",0.888889
3,Red King's BBQ,3.5,"barbeque, chinese, restaurants, juice bars & s...",0.777778
8,Wong Wong Chinese Restaurant,3.5,"chinese, restaurants",0.777778
9,Chi Ken,3.5,"asian fusion, taiwanese, restaurants, chicken ...",0.777778
0,Rising Tide,3.0,"seafood, restaurants, nightlife, bars, asian f...",0.666667


# Using all values combined for hybrid based recommendation

In [110]:
combined, df_content, df_cf, df_gnn, df_loc, df_pop =hybrid_recommend(user_id=user_id_test[0], name=names[1], description=descriptions[1], lat=lat_lon_pairs[0][0], lon=lat_lon_pairs[0][1], top_n=12)

print(f"User ID: {user_id_test[0]} | Name: {names[1]} | Descr: {descriptions[1][:23]}... | Lat: {lat_lon_pairs[0][0]} | Lon: {lat_lon_pairs[0][1]}")



/tmp/ipykernel_2607053/3261615286.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  restaurant_df['distance'] = restaurant_df.apply(


User ID: Hp3F4zLIpCc2QZ5QxB2U2Q | Name: The Gumbo Krewe | Descr: southern, salad, specia... | Lat: 39.9555052 | Lon: -75.1555641


In [111]:
combined

,name,stars,categories,score_norm
12,Yolk White & Associates,5.0,"food stands, restaurants, food, coffee & tea, ...",1.0
13,HopScotch Cafe,5.0,"nightlife, bars, beer bar, restaurants, sandwi...",1.0
5,Hair Cut & Color Design by Carleen Sanchez,5.0,"shopping, cosmetics & beauty supply, hair exte...",1.0
6,Novella Wine Bar,5.0,"nightlife, cafes, bars, food, wine bars, resta...",1.0
16,Mio’s Grill & Cafe,5.0,"beer bar, vegetarian, greek, bars, mediterrane...",1.0
15,The Liquor & Wine Grotto,5.0,"food, beer, wine & spirits",1.0
10,Delicious Expressions,5.0,"caterers, restaurants, desserts, bakeries, eve...",1.0
11,20 Shekels Bread,5.0,"bakeries, coffee & tea, breakfast & brunch, or...",1.0
28,Ruby Slipper - New Orleans,4.5,"restaurants, american (traditional), american ...",0.9
21,Hattie B’s Hot Chicken - Nashville,4.5,"american (traditional), chicken shop, southern...",0.9
